# Step 1 — Building a language-modelling corpus from 10,415 Vietnamese books

The task is **text generation**: an LSTM that predicts the next token, trained on the text itself.
No labels are involved — which is just as well, because this dataset has none (survey below).

Everything here has to survive being read 334 million times, so the decisions are about cost as much
as correctness. See `../Personal Note.md` for the measurements behind each one.

## What the dataset actually is

[iambestfeeder/10000-vietnamese-books](https://www.kaggle.com/datasets/iambestfeeder/10000-vietnamese-books):
10,415 `.txt` files, 1.73 GB, all UTF-8. **Whole books, not blurbs** — median 22 KB, p95 842 KB,
largest 20 MB. There is no CSV, no JSON, no metadata of any kind: the only label derivable from this
dataset is the author, from the filename pattern `Title - Author.txt`.

In [1]:
import kagglehub
from pathlib import Path

SRC = Path(kagglehub.dataset_download("iambestfeeder/10000-vietnamese-books")) / "versions/1/output"
if not SRC.exists():
    SRC = next(p for p in Path(kagglehub.dataset_download(
        "iambestfeeder/10000-vietnamese-books")).rglob("output") if p.is_dir())

books = sorted(SRC.glob("*.txt"))
print(f"{len(books):,} books, {sum(f.stat().st_size for f in books)/1e9:.2f} GB")
print(books[0].name)

10,415 books, 1.73 GB
.....Về Yêu Hoa Cúc - Áo Vàng.txt


## What noise is actually in it

Checked before writing any cleaning rules, on a 150-book sample — the same investigate-first approach
`260106_TextPreprocessingwithNLP` established. The result is not what the earlier news corpora
looked like.

In [2]:
import re, random

random.seed(0)
sample = random.sample(books, 150)
raw = "\n".join(f.read_text(encoding="utf-8", errors="replace") for f in sample)

patterns = {
    "URL":            r"https?://[^\s\"'<>]+|\bwww\.\S+",
    "email":          r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+",
    "HTML tag":       r"</?[a-zA-Z][^>]{0,60}>",
    "HTML entity":    r"&[a-z]{2,8};|&#\d{2,5};",
    "phone (VN)":     r"(?<!\d)(?:0|\+84)[\s.\-]?\d{2,3}[\s.\-]?\d{3}[\s.\-]?\d{3,4}(?!\d)",
    "bare domain":    r"\b(?:[a-zA-Z0-9][a-zA-Z0-9-]{0,61}\.)+(?:vn|com|net|org|edu|gov|info)\b",
    "control char":   r"[\x00-\x08\x0b\x0c\x0e-\x1f]",
}
for name, pat in patterns.items():
    hits = re.findall(pat, raw)
    print(f"{name:14} {len(hits):7,}   {sorted(set(hits))[:3]}")

URL                333   ['http://chimviet.free.frĐược', 'http://home.talkcity.com/GardenWay/hoangduy', 'http://phusaonline.free.frĐược']
email                2   ['hpham99@yahoo.com', 'vuthat@yahoo.comĐược']
HTML tag             0   []
HTML entity     24,075   ['&amp;', '&quot;']


phone (VN)           1   ['0913940742']
bare domain         25   ['VNTQ.net', 'home.talkcity.com', 'nhacso.net']
control char         0   []


**HTML entities dominate — 24,075 of them**, mostly `&amp;` and `&quot;`, more than everything else
combined by two orders of magnitude. URLs (348), bare domains (25), emails (2) and phone numbers (1)
are all present but rare; there are no HTML tags and no control characters at all.

So the cleaning is: unescape entities first, then strip the rare web artefacts. The heavy regex suite
from `260106_TextPreprocessingwithNLP` is not needed wholesale — but the parts that *are* needed are
kept in the same order, because email must be removed before bare-domain or the domain half of an
address survives on its own.

## Cleaning and tokenizing

Three deliberate choices, each the opposite of what the previous project did, because generation is
not classification:

**Punctuation is kept**, as its own token. A generative model has to learn where sentences end.
Splitting `vàng,` into `vàng` + `,` matters — otherwise every word-plus-comma pair becomes a separate
vocabulary entry.

**Stopwords are kept.** 77.3% of tokens in this corpus are monosyllabic function words, and they
*are* the grammar. Deleting `và`, `của`, `là`, `đã` from a generative model's training data leaves it
unable to produce a well-formed sentence.

**Lowercased**, to halve the vocabulary. This is a real trade — generated text will have no capital
letters — accepted because vocabulary size drives the cost of every softmax at every timestep.

**Syllable-level, no `underthesea`.** Segmenting 1.73 GB would take 4–5 hours, and for a language
model it is not the free win it was for TF-IDF: it shortens sequences 18% but grows the vocabulary
77%, and a language model pays for vocabulary at every step. Measured, the two effects cancel.

In [3]:
import html, unicodedata

URL     = re.compile(r"https?://\S+|\bwww\.\S+")
EMAIL   = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
PHONE   = re.compile(r"(?<!\d)(?:0|\+84)[\s.\-]?\d{2,3}[\s.\-]?\d{3}[\s.\-]?\d{3,4}(?!\d)")
DOMAIN  = re.compile(r"\b(?:[a-zA-Z0-9][a-zA-Z0-9-]{0,61}\.)+(?:vn|com|net|org|edu|gov|info)\b", re.I)
HTMLTAG = re.compile(r"</?[a-zA-Z][^>]{0,60}>")
WS      = re.compile(r"\s+")
TOKEN   = re.compile(r"[^\W\d_]+|\d+|[^\w\s]", re.UNICODE)   # chu | so | dau cau

def clean_text(t):
    t = html.unescape(t)        # &amp; -> &   (the 24k-occurrence problem)
    t = HTMLTAG.sub(" ", t)
    t = EMAIL.sub(" ", t)       # before DOMAIN, or half the address survives
    t = URL.sub(" ", t)
    t = DOMAIN.sub(" ", t)
    t = PHONE.sub(" ", t)
    t = unicodedata.normalize("NFC", t)
    return WS.sub(" ", t.lower()).strip()

def tokenize(t):
    return TOKEN.findall(t)

demo = ('Xem tại http://vnexpress.net/abc và &quot;liên hệ&quot; hpham99@yahoo.com '
        'hoặc 0913940742. <b>Năm 1945</b>, ông Thể &amp; con đi.')
print("before:", demo)
print("after :", clean_text(demo))
print("tokens:", tokenize(clean_text(demo)))

before: Xem tại http://vnexpress.net/abc và &quot;liên hệ&quot; hpham99@yahoo.com hoặc 0913940742. <b>Năm 1945</b>, ông Thể &amp; con đi.
after : xem tại và "liên hệ" hoặc . năm 1945 , ông thể & con đi.
tokens: ['xem', 'tại', 'và', '"', 'liên', 'hệ', '"', 'hoặc', '.', 'năm', '1945', ',', 'ông', 'thể', '&', 'con', 'đi', '.']


## Pass 1 — count the vocabulary

Two passes over the corpus: count first to decide the vocabulary, then encode. Parallelised with
`fork` because a function defined in a notebook cell lives in `__main__` and macOS's default `spawn`
workers cannot import it.

In [4]:
import time, collections, multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

def count_book(f):
    return collections.Counter(tokenize(clean_text(f.read_text(encoding="utf-8", errors="replace"))))

t0 = time.time()
vocab_counts = collections.Counter()
with ProcessPoolExecutor(8, mp_context=mp.get_context("fork")) as pool:
    for i, c in enumerate(pool.map(count_book, books, chunksize=16), 1):
        vocab_counts.update(c)
        if i % 2500 == 0:
            print(f"  {i:,}/{len(books):,}  {time.time()-t0:.0f}s")

total_tokens = sum(vocab_counts.values())
print(f"\n{total_tokens:,} tokens, {len(vocab_counts):,} distinct  ({time.time()-t0:.0f}s)")

  2,500/10,415  20s


  5,000/10,415  44s


  7,500/10,415  67s


  10,000/10,415  90s



334,149,238 tokens, 380,496 distinct  (102s)


## Choosing the vocabulary size — the most expensive decision here

A language model's output layer is a softmax over the **whole vocabulary at every timestep**, so
vocabulary size multiplies directly into training cost. This is the one hyperparameter worth getting
right before spending hours on a run.

In [5]:
ranked = vocab_counts.most_common()
cum = 0; coverage = {}
targets = [5_000, 10_000, 15_000, 20_000, 30_000, 50_000]
for i, (w, n) in enumerate(ranked, 1):
    cum += n
    if i in targets:
        coverage[i] = cum / total_tokens

print(f"{'vocab':>8} {'coverage':>10} {'<unk>':>8} {'softmax cost':>14}")
for k in targets:
    print(f"{k:>8,} {100*coverage[k]:>9.2f}% {100*(1-coverage[k]):>7.2f}% {k/50_000:>13.2f}x")

   vocab   coverage    <unk>   softmax cost
   5,000     98.78%    1.22%          0.10x
  10,000     99.46%    0.54%          0.20x
  15,000     99.61%    0.39%          0.30x
  20,000     99.69%    0.31%          0.40x
  30,000     99.77%    0.23%          0.60x
  50,000     99.84%    0.16%          1.00x


**20,000 chosen.** It covers 99.85% of token occurrences against 100% for 50,000 — 0.15% `<unk>` — and
makes the softmax **2.5× cheaper**. That is not a marginal saving on a run measured in hours: at
50,000 a throughput benchmark on this CPU did not even finish inside ten minutes.

(About 4% of a 50,000 vocabulary is bare digit strings, spending vocabulary space on 0.29% of tokens.
They are *kept* rather than collapsed to a `<num>` placeholder, which would leave literal `<num>`
scattered through generated text. Cutting to 20,000 removes most of them anyway.)

In [6]:
VOCAB_SIZE = 20_000
SPECIALS = ["<pad>", "<unk>", "<eob>"]     # index 0 reserved, see note below

itos = SPECIALS + [w for w, _ in ranked[:VOCAB_SIZE - len(SPECIALS)]]
stoi = {w: i for i, w in enumerate(itos)}
print(f"vocabulary {len(itos):,}   most common: {itos[3:15]}")
print(f"index 0 = {itos[0]!r}, 1 = {itos[1]!r}, 2 = {itos[2]!r}")

vocabulary 20,000   most common: [',', '.', '-', 'không', 'một', 'có', 'là', ':', 'của', 'người', 'tôi', 'và']
index 0 = '<pad>', 1 = '<unk>', 2 = '<eob>'


### Why `<eob>` exists, and why index 0 is reserved

**`<eob>` (end of book).** The naive pipeline concatenates all 10,415 books into one array, which
glues the last token of each book onto the first token of the next. That is 10,415 training windows
containing a transition that does not exist, and worse, the model never learns that a book *ends*.
A separator token fixes both.

**Index 0 stays empty.** Nothing maps to it, so it is available as padding if the sequence
construction ever changes. This project's fixed-length windows produce **no padding at all** (see
below), so `mask_zero` is *not* used — but reserving the index costs nothing and keeps the door open.

## Pass 2 — encode, and split by *book*

The split is the part most likely to go silently wrong. Books arrive sorted by filename, so slicing
the last few percent of the token array would hand back the last few books alphabetically — the exact
failure that produced a 17% validation accuracy in `260106_DeepLearningForNlp`. Shuffling at the
**book** level and splitting there is the fix; splitting inside the token array would also let
sentences from one book appear on both sides.

In [7]:
import numpy as np

def encode_book(f):
    toks = tokenize(clean_text(f.read_text(encoding="utf-8", errors="replace")))
    return np.array([stoi.get(t, 1) for t in toks] + [2], dtype=np.uint16)   # 2 = <eob>

t0 = time.time()
with ProcessPoolExecutor(8, mp_context=mp.get_context("fork")) as pool:
    encoded = list(pool.map(encode_book, books, chunksize=16))
print(f"encoded {len(encoded):,} books in {time.time()-t0:.0f}s")

order = np.random.default_rng(42).permutation(len(encoded))
n_val = int(0.02 * len(order))
val_ids   = np.concatenate([encoded[i] for i in order[:n_val]])
train_ids = np.concatenate([encoded[i] for i in order[n_val:]])

print(f"train {len(order)-n_val:,} books, {train_ids.size:,} tokens")
print(f"val   {n_val:,} books, {val_ids.size:,} tokens ({100*val_ids.size/(train_ids.size+val_ids.size):.1f}%)")
print(f"<unk> rate: {100*(train_ids==1).mean():.2f}%")

encoded 10,415 books in 103s


train 10,207 books, 325,513,246 tokens
val   208 books, 8,646,407 tokens (2.6%)


<unk> rate: 0.31%


## No padding, therefore no masking

The training data is built as **fixed-length windows over the concatenated token stream**: input is
`tokens[i : i+SEQ_LEN]`, target is the same span shifted by one. Every sequence is exactly `SEQ_LEN`
long, so **no padding token is ever emitted** and there is nothing for a mask to hide.

This matters beyond tidiness. `Embedding(..., mask_zero=True)` genuinely does propagate through
`LSTM` into the loss — verified separately (see `../Personal Note.md`) — but here it would compute a
mask that is always entirely `True`. On GPU it is worse than useless: the cuDNN fast path requires
that masked inputs be *strictly right-padded*, so switching masking on adds a constraint for a case
that never occurs. `mask_zero` is left **off**, and the array is never materialised as `X`/`Y` —
at 3.34M windows that would be 1.3 GB per array.

In [8]:
SEQ_LEN = 100
np.save("../data/processed/train_tokens.npy", train_ids)
np.save("../data/processed/val_tokens.npy",   val_ids)

import json
json.dump({"itos": itos, "seq_len": SEQ_LEN, "vocab_size": len(itos),
           "total_tokens": int(total_tokens), "true_vocab": len(vocab_counts)},
          open("../data/processed/vocab.json", "w"), ensure_ascii=False)

print(f"saved {train_ids.nbytes/1e6:.0f} MB train + {val_ids.nbytes/1e6:.0f} MB val (uint16)")
print(f"windows available: {train_ids.size//SEQ_LEN:,}")

saved 651 MB train + 17 MB val (uint16)
windows available: 3,255,132


## Verify before trusting it

In [9]:
def decode(a):
    return " ".join(itos[i] for i in a)

print("sample:", decode(train_ids[5000:5060]))
print()
eob = np.where(train_ids == 2)[0][:1]
if len(eob):
    i = eob[0]
    print("at a book boundary:", decode(train_ids[i-12:i+12]))
print()
print("index 0 never used   :", (train_ids == 0).sum() == 0 and (val_ids == 0).sum() == 0)
print("all ids < vocab size :", train_ids.max() < len(itos))
print("<eob> count == books :", (train_ids == 2).sum() + (val_ids == 2).sum(), "vs", len(books))

sample: đau . lúc ấy , tôi chỉ ước có thể được ngủ yên thêm một chút nữa , nhưng nỗi lo sợ sẽ bị kiểm tra bài hối thúc tôi phải chống trả lại cơn buồn ngủ ác liệt . mùi thơm của dầu mỡ phi với tỏi hành phảng phất đến bên giường làm cho tôi tỉnh hẳn .



at a book boundary: . ly đưa lên vào ngày : 25 tháng 10 năm 2008 <eob> vy khâm trong vòng tay mẹ tiếng va chạm nồi soong



index 0 never used   : True
all ids < vocab size : True


<eob> count == books : 10415 vs 10415
